In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# READ CLEAN SILVER
# ============================================================

df = spark.table("job_market.silver.jobs_clean")

total_jobs = df.count()

print("Clean Silver records:", total_jobs)

display(
    df.select(
        "job_id",
        "company",
        "job_title",
        "location",
        "location_normalized",
        "workplace_type",
        "is_remote"
    ).limit(20)
)

Clean Silver records: 2221


job_id,company,job_title,location,location_normalized,workplace_type,is_remote
5037833004,GuidePoint Security,GPSU Military/Military Spouses - Commercial Fellowship,Remote,remote,remote,true
5037876004,GuidePoint Security,GPSU Military - Federal Fellowship (Hybrid),"Reston, VA","reston, va",null,false
5390243,Cloudflare,"Senior Software Engineer, Durable Objects (DO)",In-Office,in-office,null,false
5415515004,Figma,"Account Executive, Mid-Market (London, United Kingdom)","London, England","london, england",null,false
5416444,Stripe,Data Analyst,Canada,canada,null,false
5422236004,Figma,"Account Executive, Mid-Market","San Francisco, CA • New York, NY • United States","san francisco, ca • new york, ny • united states",null,false
5426468004,Figma,"Account Executive, Enterprise","San Francisco, CA • New York, NY • United States","san francisco, ca • new york, ny • united states",null,false
5428656004,Figma,"Account Executive, Enterprise (Paris, France)","Paris, France","paris, france",null,false
5458801004,Figma,"Account Executive, Enterprise (London, United Kingdom)","London, England","london, england",null,false
5551532004,Figma,Software Engineer - Machine Learning,"San Francisco, CA • New York, NY • United States","san francisco, ca • new york, ny • united states",null,false


In [0]:
# ============================================================
# PREPARE LOCATION DATA
# ============================================================

location_base = (
    df
    .withColumn(
        "location_display",
        F.trim(
            F.regexp_replace(
                F.col("location"),
                r"\s+",
                " "
            )
        )
    )
    .withColumn(
        "location_normalized_clean",
        F.lower(
            F.trim(
                F.regexp_replace(
                    F.coalesce(
                        F.col("location_normalized"),
                        F.lit("unknown")
                    ),
                    r"\s+",
                    " "
                )
            )
        )
    )
)

display(
    location_base.select(
        "location_display",
        "location_normalized_clean",
        "company",
        "job_title"
    ).limit(20)
)

location_display,location_normalized_clean,company,job_title
Remote,remote,GuidePoint Security,GPSU Military/Military Spouses - Commercial Fellowship
"Reston, VA","reston, va",GuidePoint Security,GPSU Military - Federal Fellowship (Hybrid)
In-Office,in-office,Cloudflare,"Senior Software Engineer, Durable Objects (DO)"
"London, England","london, england",Figma,"Account Executive, Mid-Market (London, United Kingdom)"
Canada,canada,Stripe,Data Analyst
"San Francisco, CA • New York, NY • United States","san francisco, ca • new york, ny • united states",Figma,"Account Executive, Mid-Market"
"San Francisco, CA • New York, NY • United States","san francisco, ca • new york, ny • united states",Figma,"Account Executive, Enterprise"
"Paris, France","paris, france",Figma,"Account Executive, Enterprise (Paris, France)"
"London, England","london, england",Figma,"Account Executive, Enterprise (London, United Kingdom)"
"San Francisco, CA • New York, NY • United States","san francisco, ca • new york, ny • united states",Figma,Software Engineer - Machine Learning


In [0]:
# ============================================================
# LOCATION DEMAND
# ============================================================

location_demand = (
    location_base
    .groupBy(
        "location_normalized_clean"
    )
    .agg(

        # Total jobs
        F.count("*").alias(
            "job_count"
        ),

        # Companies hiring
        F.countDistinct(
            "company"
        ).alias(
            "company_count"
        ),

        # Unique job titles
        F.countDistinct(
            "job_title"
        ).alias(
            "job_title_count"
        ),

        # Remote jobs
        F.sum(
            F.when(
                F.col("is_remote") == True,
                1
            ).otherwise(0)
        ).alias(
            "remote_job_count"
        ),

        # Hybrid jobs
        F.sum(
            F.when(
                F.lower(
                    F.col("workplace_type")
                ) == "hybrid",
                1
            ).otherwise(0)
        ).alias(
            "hybrid_job_count"
        ),

        # Onsite jobs
        F.sum(
            F.when(
                F.lower(
                    F.col("workplace_type")
                ) == "onsite",
                1
            ).otherwise(0)
        ).alias(
            "onsite_job_count"
        ),

        # First posting
        F.min(
            "posted_at"
        ).alias(
            "earliest_posted_at"
        ),

        # Latest posting
        F.max(
            "posted_at"
        ).alias(
            "latest_posted_at"
        )
    )
)

In [0]:
# ============================================================
# LOCATION DISPLAY NAME
# ============================================================

location_lookup = (
    location_base
    .filter(
        F.col("location_normalized_clean") != "unknown"
    )
    .groupBy(
        "location_normalized_clean"
    )
    .agg(
        F.min(
            "location_display"
        ).alias(
            "location"
        )
    )
)

location_demand = (
    location_demand
    .join(
        location_lookup,
        on="location_normalized_clean",
        how="left"
    )
    .withColumn(
        "location",
        F.coalesce(
            F.col("location"),
            F.lit("Unknown")
        )
    )
)

In [0]:
# ============================================================
# LOCATION DEMAND RANKING
# ============================================================

location_rank_window = Window.orderBy(
    F.desc("job_count"),
    F.asc("location_normalized_clean")
)

location_demand = (
    location_demand
    .withColumn(
        "location_demand_rank",
        F.dense_rank().over(
            location_rank_window
        )
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
# ============================================================
# FINAL LOCATION DEMAND DATASET
# ============================================================

location_demand = (
    location_demand
    .withColumn(
        "demand_percentage",
        F.round(
            (
                F.col("job_count") /
                F.lit(total_jobs)
            ) * 100,
            2
        )
    )
    .withColumn(
        "remote_percentage",
        F.round(
            (
                F.col("remote_job_count") /
                F.col("job_count")
            ) * 100,
            2
        )
    )
    .withColumn(
        "hybrid_percentage",
        F.round(
            (
                F.col("hybrid_job_count") /
                F.col("job_count")
            ) * 100,
            2
        )
    )
    .withColumn(
        "onsite_percentage",
        F.round(
            (
                F.col("onsite_job_count") /
                F.col("job_count")
            ) * 100,
            2
        )
    )
    .select(
        "location_demand_rank",
        "location",
        "location_normalized_clean",
        "job_count",
        "demand_percentage",
        "company_count",
        "job_title_count",
        "remote_job_count",
        "remote_percentage",
        "hybrid_job_count",
        "hybrid_percentage",
        "onsite_job_count",
        "onsite_percentage",
        "earliest_posted_at",
        "latest_posted_at"
    )
    .orderBy(
        F.asc("location_demand_rank")
    )
)

display(
    location_demand.limit(30)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


location_demand_rank,location,location_normalized_clean,job_count,demand_percentage,company_count,job_title_count,remote_job_count,remote_percentage,hybrid_job_count,hybrid_percentage,onsite_job_count,onsite_percentage,earliest_posted_at,latest_posted_at
1,Hybrid,hybrid,311,14.0,1,300,0,0.0,0,0.0,0,0.0,2025-01-15T21:09:18.000Z,2026-10-06T10:09:40.000Z
2,"Foster City, CA","foster city, ca",194,8.73,1,194,0,0.0,136,70.1,58,29.9,2016-02-08T19:02:13.000Z,2026-10-05T22:22:13.000Z
3,"New York, NY","new york, ny",134,6.03,4,134,19,14.18,75,55.97,30,22.39,2014-01-23T00:00:00.000Z,2026-10-02T18:36:55.000Z
4,Remote - USA,remote - usa,134,6.03,1,132,134,100.0,0,0.0,0,0.0,2025-12-19T17:17:18.000Z,2026-10-02T21:44:26.000Z
5,"San Francisco, CA • New York, NY • United States","san francisco, ca • new york, ny • united states",89,4.01,1,89,0,0.0,0,0.0,0,0.0,2025-01-22T14:27:24.000Z,2026-10-05T20:40:56.000Z
6,"Washington, D.C.","washington, d.c.",72,3.24,1,72,0,0.0,36,50.0,36,50.0,2018-06-28T16:26:03.000Z,2026-09-25T15:29:56.000Z
7,London,london,56,2.52,2,56,11,19.64,13,23.21,1,1.79,2025-10-22T14:26:18.000Z,2026-10-05T12:48:25.000Z
8,Singapore,singapore,56,2.52,4,56,0,0.0,0,0.0,1,1.79,2026-01-13T06:12:25.000Z,2026-10-01T02:15:03.000Z
9,Distributed,distributed,52,2.34,1,49,0,0.0,0,0.0,0,0.0,2025-07-11T22:15:05.000Z,2026-10-05T15:18:12.000Z
10,Remote,remote,50,2.25,3,50,50,100.0,0,0.0,0,0.0,2023-12-13T16:52:37.000Z,2026-10-05T15:16:04.000Z


In [0]:
# ============================================================
# LOCATION DEMAND VALIDATION
# ============================================================

gold_location_job_count = (
    location_demand
    .agg(
        F.sum(
            "job_count"
        ).alias(
            "total_job_count"
        )
    )
    .collect()[0]["total_job_count"]
)

print("======================================")
print("LOCATION DEMAND VALIDATION")
print("======================================")
print("Silver records :", total_jobs)
print("Gold job count :", gold_location_job_count)
print(
    "Difference     :",
    total_jobs - gold_location_job_count
)
print("======================================")

assert total_jobs == gold_location_job_count, \
    "Location demand aggregation does not reconcile!"

print("✓ Location demand reconciliation passed.")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


LOCATION DEMAND VALIDATION
Silver records : 2221
Gold job count : 2221
Difference     : 0
✓ Location demand reconciliation passed.


In [0]:
# ============================================================
# SAVE LOCATION DEMAND GOLD TABLE
# ============================================================

location_demand.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "overwriteSchema",
        "true"
    ) \
    .saveAsTable(
        "job_market.gold.location_demand"
    )

print("✓ Created:")
print("job_market.gold.location_demand")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✓ Created:
job_market.gold.location_demand


In [0]:
# ============================================================
# LOCATION × JOB ROLE DEMAND
# ============================================================

location_job_demand = (
    location_base
    .groupBy(
        "location_normalized_clean",
        "job_title"
    )
    .agg(
        F.count("*").alias(
            "job_count"
        ),

        F.countDistinct(
            "company"
        ).alias(
            "company_count"
        ),

        F.sum(
            F.when(
                F.col("is_remote") == True,
                1
            ).otherwise(0)
        ).alias(
            "remote_job_count"
        )
    )
)

location_job_demand = (
    location_job_demand
    .join(
        location_lookup,
        on="location_normalized_clean",
        how="left"
    )
    .withColumn(
        "location",
        F.coalesce(
            F.col("location"),
            F.lit("Unknown")
        )
    )
    .select(
        "location",
        "location_normalized_clean",
        "job_title",
        "job_count",
        "company_count",
        "remote_job_count"
    )
    .orderBy(
        F.desc("job_count")
    )
)

display(
    location_job_demand.limit(30)
)

location,location_normalized_clean,job_title,job_count,company_count,remote_job_count
Hybrid,hybrid,Senior Forward Deployed Engineer,5,1,0
In-Office,in-office,Senior Data Engineer,2,1,0
N/A,n/a,"Full Stack Engineer, Enterprise Engineering",2,1,0
"Hybrid - London, UK","hybrid - london, uk",Concierge Specialist IV,2,1,0
US-Remote,us-remote,"Communities Partner Development Manager, SaaS Platforms",2,1,2
"Toronto, Canada","toronto, canada","Account Executive, SMB Grower",2,1,0
Hybrid,hybrid,"Senior Customer Engineer, Public Sector Federal",2,1,0
Hybrid - Singapore,hybrid - singapore,Concierge Specialist IV,2,1,0
Distributed,distributed,"Director, Growth Sales (AI & Digital Natives)",2,1,0
Remote - USA,remote - usa,Senior Software Engineer,2,1,2


In [0]:
# ============================================================
# SAVE LOCATION × JOB ROLE
# ============================================================

location_job_demand.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "overwriteSchema",
        "true"
    ) \
    .saveAsTable(
        "job_market.gold.location_job_demand"
    )

print("✓ Created:")
print("job_market.gold.location_job_demand")

✓ Created:
job_market.gold.location_job_demand


In [0]:
# ============================================================
# WORKPLACE TYPE SUMMARY
# ============================================================

workplace_demand = (
    df
    .withColumn(
        "workplace_category",
        F.when(
            F.col("is_remote") == True,
            "Remote"
        )
        .when(
            F.lower(
                F.col("workplace_type")
            ) == "hybrid",
            "Hybrid"
        )
        .when(
            F.lower(
                F.col("workplace_type")
            ) == "onsite",
            "Onsite"
        )
        .otherwise(
            "Not Specified"
        )
    )
    .groupBy(
        "workplace_category"
    )
    .agg(
        F.count("*").alias(
            "job_count"
        ),
        F.countDistinct(
            "company"
        ).alias(
            "company_count"
        )
    )
    .withColumn(
        "demand_percentage",
        F.round(
            (
                F.col("job_count") /
                F.lit(total_jobs)
            ) * 100,
            2
        )
    )
    .orderBy(
        F.desc("job_count")
    )
)

display(workplace_demand)

workplace_category,job_count,company_count,demand_percentage
Not Specified,1234,5,55.56
Remote,394,6,17.74
Hybrid,392,4,17.65
Onsite,201,3,9.05


In [0]:
# ============================================================
# SAVE WORKPLACE DEMAND
# ============================================================

workplace_demand.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "overwriteSchema",
        "true"
    ) \
    .saveAsTable(
        "job_market.gold.workplace_demand"
    )

print("✓ Created:")
print("job_market.gold.workplace_demand")

✓ Created:
job_market.gold.workplace_demand


In [0]:
# ============================================================
# FINAL GOLD VALIDATION
# ============================================================

gold_location = spark.table(
    "job_market.gold.location_demand"
)

gold_location_job = spark.table(
    "job_market.gold.location_job_demand"
)

gold_workplace = spark.table(
    "job_market.gold.workplace_demand"
)

print("======================================")
print("LOCATION DEMAND GOLD VALIDATION")
print("======================================")

print(
    "Clean Silver jobs:",
    total_jobs
)

print(
    "Location demand rows:",
    gold_location.count()
)

print(
    "Location × Job rows:",
    gold_location_job.count()
)

print(
    "Workplace summary rows:",
    gold_workplace.count()
)

print("======================================")

print("✓ Location Demand Gold layer completed.")

LOCATION DEMAND GOLD VALIDATION
Clean Silver jobs: 2221
Location demand rows: 351
Location × Job rows: 2194
Workplace summary rows: 4
✓ Location Demand Gold layer completed.
